In [1]:
!pip install transformers datasets sentencepiece torch torchvision torchaudio
!pip install scikit-learn pandas matplotlib seaborn

In [ ]:
pip install tf-keras

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import BartTokenizer, BartForConditionalGeneration, Trainer, TrainingArguments
from transformers import DataCollatorForSeq2Seq
import torch


In [ ]:
def load_bbc_dataset(base_path=r"D:\news summary\BBC News Summary\BBC News Summary"):
    articles, summaries, categories = [], [], []

    news_path = os.path.join(base_path, "News Articles")
    sum_path = os.path.join(base_path, "Summaries")

    for category in os.listdir(news_path):
        article_dir = os.path.join(news_path, category)
        summary_dir = os.path.join(sum_path, category)

        for fname in os.listdir(article_dir):
            art_file = os.path.join(article_dir, fname)
            sum_file = os.path.join(summary_dir, fname)

            if os.path.exists(sum_file):
                with open(art_file, "r", encoding="latin-1") as af:
                    article = af.read().strip()
                with open(sum_file, "r", encoding="latin-1") as sf:
                    summary = sf.read().strip()

                articles.append(article)
                summaries.append(summary)
                categories.append(category)

    return pd.DataFrame({"category": categories, "article": articles, "summary": summaries})

df = load_bbc_dataset()
print(df.head())
print("Total samples:", len(df))


In [ ]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
print("Train size:", len(train_df), "Test size:", len(test_df))


In [ ]:
# Cell 5: tokenizer & model
from transformers import BartTokenizer, BartForConditionalGeneration

model_name = "facebook/bart-base"   # small-ish; change to bart-large-cnn if you have GPU
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

print("Loaded tokenizer & model:", model_name)


In [ ]:
# Cell 6: convert pandas -> datasets and tokenize
from datasets import Dataset
import numpy as np

# ensure indices won't become a column
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

# create HF Datasets
train_ds = Dataset.from_pandas(train_df)
eval_ds  = Dataset.from_pandas(test_df)

# Tokenize function (no pre-padding; let data collator pad dynamically)
max_input_length = 512
max_target_length = 128

def tokenize_function(batch):
    # batch['article'] and batch['summary'] are lists when batched=True
    model_inputs = tokenizer(batch["article"], max_length=max_input_length, truncation=True)
    # Tokenize targets (as labels)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(batch["summary"], max_length=max_target_length, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# map tokenization (batched)
train_tokenized = train_ds.map(tokenize_function, batched=True, remove_columns=train_ds.column_names)
eval_tokenized  = eval_ds.map(tokenize_function, batched=True, remove_columns=eval_ds.column_names)

print("Tokenized train examples:", len(train_tokenized), "eval examples:", len(eval_tokenized))


In [ ]:
pip install rouge_score

In [ ]:
# Cell 7: data collator + rouge metric + helper functions
from transformers import DataCollatorForSeq2Seq
import evaluate
import numpy as np

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

rouge = evaluate.load("rouge")

def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [label.strip() for label in labels]
    return preds, labels

def compute_metrics(eval_pred):
    """
    eval_pred is (preds, labels). When predict_with_generate=True, preds may be a tuple.
    """
    preds_ids, labels_ids = eval_pred
    # if preds is tuple (generated, something), get first element
    if isinstance(preds_ids, tuple):
        preds_ids = preds_ids[0]
    decoded_preds = tokenizer.batch_decode(preds_ids, skip_special_tokens=True)
    # Replace -100 in labels (if present) with pad token id for decoding
    labels_ids = np.where(labels_ids != -100, labels_ids, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels_ids, skip_special_tokens=True)
    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    # keep only mid fmeasure scores and format
    result = {k: v.mid.fmeasure * 100 for k, v in result.items()}
    result = {k: round(v, 4) for k, v in result.items()}
    return result


In [ ]:
pip install transformers[torch]

In [ ]:
!pip install --upgrade accelerate transformers[torch]


In [ ]:
# Cell 8: training args + Trainer
from transformers import TrainingArguments, Trainer
import torch

# check device
device = 0 if torch.cuda.is_available() else -1
print("CUDA available:", torch.cuda.is_available(), "using device:", device)

training_args = TrainingArguments(
    output_dir="./results",
    #evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=2,   # reduce to 1 if OOM
    per_device_eval_batch_size=2,
    num_train_epochs=1,             # increase later
    weight_decay=0.01,
    save_total_limit=2,
   # predict_with_generate=True,     # required to compute generation metrics like ROUGE
    logging_dir="./logs",
    logging_steps=100,
    fp16=torch.cuda.is_available(), # use fp16 if CUDA available
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=eval_tokenized,
    processing_class=tokenizer,   # 👈 replace tokenizer=... with this
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer created.")


In [ ]:
# Cell 9: train
# WARNING: may be slow on CPU. For quick debug reduce dataset size by .select(...)
# Example to train on a small subset for quick tests:
# small_train = train_tokenized.select(range(200))
# small_eval  = eval_tokenized.select(range(50))
# Then pass small_train / small_eval to the Trainer constructor.

trainer.train()
trainer.save_model("./bart_finetuned_small")
tokenizer.save_pretrained("./bart_finetuned_small")
print("Training complete and model saved.")


In [ ]:
trainer.train()


In [ ]:
test_text = """Labour plans maternity pay rise

Maternity pay for new mothers is to rise by £1,400 as part of new proposals announced by the Trade and Industry Secretary Patricia Hewitt.

It would mean paid leave would be increased to nine months by 2007, Ms Hewitt told GMTV's Sunday programme. Other plans include letting maternity pay be given to fathers and extending rights to parents of older children. The Tories dismissed the maternity pay plan as "desperate", while the Liberal Democrats said it was misdirected.

Ms Hewitt said: "We have already doubled the length of maternity pay, it was 13 weeks when we were elected, we have already taken it up to 26 weeks. "We are going to extend the pay to nine months by 2007 and the aim is to get it right up to the full 12 months by the end of the next Parliament." She said new mothers were already entitled to 12 months leave, but that many women could not take it as only six of those months were paid. "We have made a firm commitment. We will definitely extend the maternity pay, from the six months where it now is to nine months, that's the extra £1,400." She said ministers would consult on other proposals that could see fathers being allowed to take some of their partner's maternity pay or leave period, or extending the rights of flexible working to carers or parents of older children. The Shadow Secretary of State for the Family, Theresa May, said: "These plans were announced by Gordon Brown in his pre-budget review in December and Tony Blair is now recycling it in his desperate bid to win back women voters."

She said the Conservatives would announce their proposals closer to the General Election. Liberal Democrat spokeswoman for women Sandra Gidley said: "While mothers would welcome any extra maternity pay the Liberal Democrats feel this money is being misdirected." She said her party would boost maternity pay in the first six months to allow more women to stay at home in that time.

Ms Hewitt also stressed the plans would be paid for by taxpayers, not employers. But David Frost, director general of the British Chambers of Commerce, warned that many small firms could be "crippled" by the move. "While the majority of any salary costs may be covered by the government's statutory pay, recruitment costs, advertising costs, retraining costs and the strain on the company will not be," he said. Further details of the government's plans will be outlined on Monday. New mothers are currently entitled to 90% of average earnings for the first six weeks after giving birth, followed by £102.80 a week until the baby is six months old.
"""

inputs = tokenizer([test_text], max_length=512, truncation=True, return_tensors="pt")
summary_ids = model.generate(inputs["input_ids"], max_length=50, min_length=10, length_penalty=2.0, num_beams=4)
print("Generated Summary:", tokenizer.decode(summary_ids[0], skip_special_tokens=True))


In [19]:
model.save_pretrained("./bbc_summarizer_model")
tokenizer.save_pretrained("./bbc_summarizer_model")


('./bbc_summarizer_model\\tokenizer_config.json',
 './bbc_summarizer_model\\special_tokens_map.json',
 './bbc_summarizer_model\\vocab.json',
 './bbc_summarizer_model\\merges.txt',
 './bbc_summarizer_model\\added_tokens.json')